In [22]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.nn.functional import cross_entropy
%matplotlib inline

words = open('../names.txt', 'r').read().splitlines()  # 32033
chars = sorted(list(set(''.join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [23]:
block_size = 3


def build_dateset(words):
    X, Y = [], []

    for w in words:
        # fill 3 letters in x fill next in y
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)  # 0.0.0 on first iteration
            Y.append(ix)
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y


import random

# first we want to shuffle the words
# then we want 80% training set
# then we want 10% dev test
# then we want 10% final test

random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

train_data = words[:n1]
dev_data = words[n1:n2]
test_data = words[n2:]

Xtr, Ytr = build_dateset(train_data)
Xdev, Ydev = build_dateset(dev_data)
Xtest, Ytest = build_dateset(test_data)

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


The first problem that we encountered we had purely random weights was that the initial loss was far from the expected. The expected initial loss was an uniform destribution. Meaning:

```python
logits = torch.tensor(27*[0], dtype=float)
probs = torch.softmax(logits, dim=0)
loss = -probs.log().mean()
single_loss = -probs[1].log()
probs, loss, single_loss


# (tensor([0.0370, 0.0370, 0.0370, 0.0370, 0.0370, 0.0370, 0.0370, 0.0370, 0.0370,
#          0.0370, 0.0370, 0.0370, 0.0370, 0.0370, 0.0370, 0.0370, 0.0370, 0.0370,
#          0.0370, 0.0370, 0.0370, 0.0370, 0.0370, 0.0370, 0.0370, 0.0370, 0.0370],
#         dtype=torch.float64),
#  tensor(3.2958, dtype=torch.float64),
#  tensor(3.2958, dtype=torch.float64))
```
the ideal initial loss is: 3.29  - That's if basically all outputs are equally likely = weights at layer being all zeros

But that is not good so we set them to some small number w2*0.1 at first

# Adjust w2 and b2

In [24]:
# MLP revisited
n_embd = 10  # the dimensionality of the character embedding vectors
n_hidden = 200

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((vocab_size, n_embd),             generator=g)      # torch.Size([27, 10])
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g)      # torch.Size([30,200])
b1 = torch.randn(n_hidden,                        generator=g)      # torch.Size([200])
W2 = torch.randn((n_hidden, vocab_size),          generator=g)*0.01  # torch.Size([200,27])
b2 = torch.randn(vocab_size,                      generator=g)*0    # torch.Size([27,])

parameters = [C, W1, b1, W2, b2]

print(sum(p.nelement() for p in parameters))  # 11897
for p in parameters:
    p.requires_grad = True

11897


In [25]:
max_steps = 200_000
batch_size = 32
lossi = []

for i in range(200_000):

    ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
    Xb, Yb = Xtr[ix], Ytr[ix]  # batch X, Y

    #forward pass
    emb = C[Xb]  # 32 samples of context of 3 letters each embedded in 10 numbers
    embcat = emb.view(emb.shape[0], -1)  # concatenate the vectors
    hpreact = embcat @ W1 + b1
    h = torch.tanh(hpreact)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Yb)

    #backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    lr = 0.01 if i < 100_000 else 0.001
    for p in parameters:
        p.data += - lr * p.grad

    if i % 10_000 == 0:
        if i == 0:
            print(f'Initial loss: {i:7d} /{max_steps:7d}: {loss.item():.4f}')

        print(f'{i:7d} /{max_steps:7d}: {loss.item():.4f}')
    lossi.append(loss.log10().item())

Initial loss:       0 / 200000: 3.3221
      0 / 200000: 3.3221
  10000 / 200000: 2.2241
  20000 / 200000: 2.4473
  30000 / 200000: 2.6137
  40000 / 200000: 1.9820
  50000 / 200000: 2.5756
  60000 / 200000: 2.4735
  70000 / 200000: 2.2249
  80000 / 200000: 2.2189
  90000 / 200000: 2.1875
 100000 / 200000: 2.0905
 110000 / 200000: 2.3329
 120000 / 200000: 2.0611
 130000 / 200000: 2.5672
 140000 / 200000: 2.3687
 150000 / 200000: 2.2264
 160000 / 200000: 1.9801
 170000 / 200000: 1.8200
 180000 / 200000: 2.1082
 190000 / 200000: 2.0759


In [26]:
# now go back and multiply w2 and b2 by almost 0 and see how we approach an uniform destribution
# and run the test again

In [27]:
# test
@torch.no_grad() # this decorator disables gradient tracking
def split_loss(split):
    # x, y = {'train': (Xtr, Ytr),
    #         'val': (Xdev, Ydev),
    #         'test': (Xtest, Ytest),
    #         }[split]

    if split == 'train':
        x, y = Xtr, Ytr
    elif split == 'val':
        x, y = Xdev, Ydev
    elif split == 'test':
        x, y = Xtest, Ytest

    emb = C[x] # (total entries, block_size, n_embd)
    embcat = emb.view(emb.shape[0],-1) # concat into (N, block_size)
    h = torch.tanh(embcat @ W1 + b1) # (N, vocab_size)
    logits = h @ W2 + b2 # (N, vocab_size)
    loss = F.cross_entropy(logits, y)
    print(split, loss.item())

split_loss('train')
split_loss('val')

train 2.1683130264282227
val 2.194282054901123


Before
```
Initial loss: 0 / 200000: 27...
train 2.28
val 2.29
```

After fixing the intial loss = fixing softmax being extremly confidently wrong
```
Initial loss: 0 / 200000: 3.3221
train 2.16
val 2.19
```

In [29]:
# As you can see the loss is far improved because we have spend less training cycles on squashing this giants weights -> basically starting from better position = same cycles - better results